# 04 — Dispatching rule

When a LITHOGRAPHY machine becomes free, which waiting lot does it pick?
The rule isn't in the data, so we infer it from observed decisions.

In [2]:
import numpy as np
import pandas as pd

from wafer_twin.data import load_tables, build_process

tables = load_tables()
process = build_process(tables)
litho = process[process["WORKSHOP"] == "LITHOGRAPHY"]

In [3]:
arr = tables["lot_arrival_table"]

# if every lot gets the same delay, due date order = fab arrival order
(arr["DUE_DATE"] - arr["ARRIVAL_DATE"]).value_counts()

61 days 00:00:00    9442
61 days 01:00:00    3662
62 days 00:00:00    3510
60 days 23:00:00    1656
58 days 23:00:00    1016
60 days 00:00:00     964
59 days 23:00:00     896
59 days 00:00:00     785
Name: count, dtype: int64

## Rebuilding each decision

For a machine starting a lot at time t, the queue it chose from is every lot that:
- has entered the LITHOGRAPHY queue before t and not started yet,
- has a recipe this machine can run (`interopt_table`).

Warm-up months are skipped and 20,000 decisions are sampled.

In [4]:
# Prepare arrays
lit = litho.sort_values("QUEUE_ENTRY").reset_index(drop=True)

q_entry = lit["QUEUE_ENTRY"].to_numpy()
begin = lit["PROCESS_BEGIN_DATE"].to_numpy()
arrival = lit["ARRIVAL_DATE"].to_numpy()
machines = lit["MACHINE"].to_numpy()
recipe_code, recipe_names = pd.factorize(lit["RECIPE"])
product_code, product_names = pd.factorize(lit["PRODUCT"])

# recipes each litho machine can run
io = tables["interopt_table"]
io = io[io["WORKSHOP"] == "LITHOGRAPHY"]
recipe_index = {r: i for i, r in enumerate(recipe_names)}
machine_recipes = {
    m: np.array([recipe_index[r] for r in g["RECIPE"] if r in recipe_index])
    for m, g in io.groupby("MACHINE")
}

# nobody waits longer than this, limits the search window
max_wait = (begin - q_entry).max()

In [9]:
#Add due dates to the arrays
lit = lit.merge(tables["lot_arrival_table"][["LOT", "DUE_DATE"]], on="LOT", how="left")
due = lit["DUE_DATE"].to_numpy()

In [10]:
# Rebuild the sampled decisions
rng = np.random.default_rng(0)
after_warmup = np.flatnonzero(begin >= np.datetime64("2023-04-01"))
sample = rng.choice(after_warmup, size=20_000, replace=False)

rows = []
for i in sample:
    t = begin[i]

    # only lots that entered the queue in the last max_wait can still be waiting
    lo = np.searchsorted(q_entry, t - max_wait, side="left")
    hi = np.searchsorted(q_entry, t, side="right")
    idx = np.arange(lo, hi)

    allowed = machine_recipes[machines[i]]
    compatible = np.isin(recipe_code[idx], allowed)
    waiting = idx[((begin[idx] > t) & compatible) | (idx == i)]

    if len(waiting) < 2:
        continue  # no real choice

    # waiting lots of the same product as the chosen one
    same = waiting[product_code[waiting] == product_code[i]]

    rows.append({
        "queue_size": len(waiting),
        "chosen_compatible": recipe_code[i] in allowed,
        "fifo_queue": q_entry[i] == q_entry[waiting].min(),
        "oldest_in_fab": arrival[i] == arrival[waiting].min(),
        "edd": due[i] == due[waiting].min(),
        "product": product_code[i],
        "product_counts": np.bincount(product_code[waiting], minlength=len(product_names)),
        "n_same_product": len(same),
        "fifo_within_product": q_entry[i] == q_entry[same].min(),
        "edd_within_product": due[i] == due[same].min(),
    })

decisions = pd.DataFrame(rows)
print(f"{len(decisions):,} decisions with a real choice")

19,462 decisions with a real choice


In [11]:
# Compare the simple rules
print("chosen recipe allowed on its machine:", f"{decisions['chosen_compatible'].mean():.1%}")
print("mean queue size:", round(decisions["queue_size"].mean(), 1))
print()
print(f"picked the oldest lot in the queue: {decisions['fifo_queue'].mean():.1%}")
print(f"picked the oldest lot in the fab:   {decisions['oldest_in_fab'].mean():.1%}")
print(f"expected by pure chance:            {(1 / decisions['queue_size']).mean():.1%}")

chosen recipe allowed on its machine: 100.0%
mean queue size: 16.0

picked the oldest lot in the queue: 8.8%
picked the oldest lot in the fab:   7.4%
expected by pure chance:            10.7%


In [12]:
# Products: chosen vs expected by chance
counts = np.vstack(decisions["product_counts"])

# if the pick were random, each product would win in proportion to its share of the queue
expected = (counts / counts.sum(axis=1, keepdims=True)).sum(axis=0)
chosen = np.bincount(decisions["product"], minlength=len(product_names))

pd.DataFrame(
    {"chosen": chosen, "expected_by_chance": expected.round(0), "ratio": (chosen / expected).round(2)},
    index=product_names,
).sort_index()

,chosen,expected_by_chance,ratio
PRODUCT_0001,5900,3759.0,1.57
PRODUCT_0002,2958,5891.0,0.50
PRODUCT_0003,6964,5610.0,1.24
PRODUCT_0004,3640,4202.0,0.87


In [13]:
# Due delay by product
arr = tables["lot_arrival_table"].copy()
arr["DELAY_DAYS"] = (arr["DUE_DATE"] - arr["ARRIVAL_DATE"]).dt.total_seconds() / 86400

arr.groupby("PRODUCT")["DELAY_DAYS"].agg(["min", "median", "max"]).round(2)

,min,median,max
PRODUCT,,,
PRODUCT_0001,58.96,61.0,62.0
PRODUCT_0002,58.96,61.0,62.0
PRODUCT_0003,58.96,61.0,62.0
PRODUCT_0004,58.96,61.0,62.0


In [15]:
# Rules within the chosen product

print(f"EDD over the whole queue: {decisions['edd'].mean():.1%}")

# only decisions where the machine had to pick between lots of the same product
multi = decisions[decisions["n_same_product"] >= 2]
print(f"\n{len(multi):,} decisions with 2+ lots of the chosen product")
print(f"FIFO within the product:  {multi['fifo_within_product'].mean():.1%}")
print(f"EDD within the product:   {multi['edd_within_product'].mean():.1%}")
print(f"expected by pure chance:  {(1 / multi['n_same_product']).mean():.1%}")

EDD over the whole queue: 7.4%

17,177 decisions with 2+ lots of the chosen product
FIFO within the product:  25.5%
EDD within the product:   25.2%
expected by pure chance:  24.8%


### Takeaways :

- Due delay is 59–62 days and the same for every product: due date = arrival + 2 calendar months
  (59 days for Jan–Feb, 62 for Jul–Aug; the −1 h cases come from daylight saving time)
- A machine only sees the lots its recipes allow: 16 lots on average, not the ~90 of the whole queue
- No age-based rule is used over the whole queue: FIFO 8.8%, oldest-in-fab 7.4%, earliest due date 7.4%,
  all below pure chance (10.7%)
- Products are clearly prioritised: *PRODUCT_0001* is picked 1.57× more than by chance,
  *PRODUCT_0003* 1.24×, *PRODUCT_0004* 0.87×, *PRODUCT_0002* only 0.50×
- Same order as the LITHOGRAPHY waiting times (42 h, 61 h, 80 h, 216 h per lot):
  product priority explains who waits most
- It also explains why age-based rules score below chance: *PRODUCT_0002* lots pile up and grow old,
  so the oldest lot in the queue is often one the machine rarely picks
- Within a product the pick looks random: FIFO 25.5%, EDD 25.2%, pure chance 24.8%
- Working hypothesis: pick a product according to a priority score, then any of its lots at random